# V4 (corrected): calibration transfer across quantum models, noise families and data shift

**Built on v2's validated pipeline** (frozen ResNet-18 features → 4 PCA components → circuits trained with exact gradients; 0–π angle encoding; matched calibration/test splits), which reached AUROC 0.98 on PneumoniaMNIST.

**What this notebook tests**

| Part | Question |
| --- | --- |
| A | How does each noise type (gate depolarizing, amplitude damping, readout, shots) damage each quantum model, and does refitting repair it? |
| B | Can calibration follow **unseen** noise without new labels? Held-out noise strengths, plus an **unseen noise family** (amplitude damping) that no method was told about. Includes a **classical control**: the MLP with the same output distortion the VQC suffered. |
| C | **Label shift** (different pneumonia rates), alone and combined with noise: can hardware correction and prior correction be separated? |
| D | How small can the unlabeled batch be for moment matching? |

**Models:** VQC · data re-uploading VQC · quantum-kernel SVM (QSVM) · classical MLP · classical RBF-SVM.
**Datasets:** PneumoniaMNIST (chest X-ray) and BreastMNIST (breast ultrasound; small, so its results are exploratory).
**Primary metrics:** Brier score and NLL; ECE is secondary.

**Safety gate:** any model whose noise-free validation AUROC is below `AUROC_GATE` is excluded and reported, so a model that learned nothing can't produce misleadingly "well-calibrated" results.

**Limitations (also printed at the end):** simulated noise only; the QSVM kernel under noise uses the overlap of noisy feature states (a proxy for a hardware fidelity circuit) and ignores readout error; data splits are fixed per dataset, so seeds vary model initialisation and measurement randomness, not the split.

**Run in Colab:** `Runtime → Change runtime type → T4 GPU`, then `Runtime → Run all`. Expect about 20–40 minutes on Colab (a 12-minute run of the experiments here, plus image-feature extraction). Set `QUICK_TEST = True` first for a ~5-minute check.

In [ ]:
!pip install -q pennylane medmnist

## 1. Settings

In [ ]:
import os, time, warnings, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torchvision
import pennylane as qml
import medmnist
from medmnist import INFO
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.svm import SVC
from scipy.optimize import minimize_scalar, minimize

warnings.filterwarnings("ignore")
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)

QUICK_TEST = False

CFG = dict(
    DATASETS=["pneumoniamnist", "breastmnist"],
    IMG_SIZE=64,
    TEST_FRAC=0.15, CAL_FRAC=0.15, VAL_FRAC=0.10,
    N_QUBITS=4, N_LAYERS=3,
    EPOCHS=40, BATCH=64, LR=0.02, PATIENCE=8,
    SEEDS=[0, 1, 2],                       # use 5-10 for the paper
    QSVM_N_TRAIN=600, QSVM_C=1.0,
    AUROC_GATE=0.75,
    # Part A: one noise type at a time
    A_DEPOL=[0.005, 0.02, 0.05], A_AMP=[0.005, 0.02, 0.05], A_SHOTS=[100, 1000, 10000],
    A_READOUT=[(0.02, 0.02), (0.01, 0.08), (0.08, 0.01)],
    # Part B: drift
    FIT_P=[0.0, 0.002, 0.01, 0.03],
    HELDOUT_P=[0.001, 0.005, 0.015, 0.02, 0.05],
    UNSEEN_AMP=[0.01, 0.03], UNSEEN_BASE_P=0.002,
    N_READOUT_TRAIN=6, N_READOUT_TEST=4, READOUT_MAX=0.08, DRIFT_SHOTS=1000,
    REF=dict(p=0.002, g=0.0, e0=0.02, e1=0.02),
    REPORT_ERROR=0.2, COND_SEED=123,
    # Part C: label shift
    PRIORS=[0.3, 0.5, 0.9], JOINT=dict(p=0.015, g=0.0, e0=0.02, e1=0.05),
    # Part D
    BATCH_SIZES=[16, 32, 64, 128],
    N_BINS=15, N_BOOT=1000,
)
if QUICK_TEST:
    CFG.update(EPOCHS=5, SEEDS=[0], QSVM_N_TRAIN=150, A_DEPOL=[0.02], A_AMP=[0.02], A_SHOTS=[1000],
               A_READOUT=[(0.01, 0.08)], FIT_P=[0.0, 0.002, 0.03], HELDOUT_P=[0.005, 0.05], UNSEEN_AMP=[0.03],
               N_READOUT_TRAIN=3, N_READOUT_TEST=2, PRIORS=[0.3, 0.9], BATCH_SIZES=[32], N_BOOT=200)

NQ, NL = CFG["N_QUBITS"], CFG["N_LAYERS"]
torch.set_default_dtype(torch.float64)
CACHE = "/content/v4_cache" if os.path.isdir("/content") else "v4_cache"
os.makedirs(CACHE, exist_ok=True)
print("PennyLane", qml.__version__, "| torch", torch.__version__, "| medmnist", medmnist.__version__)

## 2. Data and frozen image features
For each dataset the official train and validation sets are pooled and re-split (stratified, fixed seed) into train / validation / calibration / test, so calibration and test share one distribution. The official test set is kept separately as the **shift set**. Features are cached, so re-running is fast.

In [ ]:
def load_split(flag, split):
    DataClass = getattr(medmnist, INFO[flag]["python_class"])
    try:
        ds = DataClass(split=split, download=True, size=CFG["IMG_SIZE"])
    except TypeError:
        ds = DataClass(split=split, download=True)
    return ds.imgs, ds.labels.ravel().astype(int)

device = "cuda" if torch.cuda.is_available() else "cpu"
encoder = torchvision.models.resnet18(weights=torchvision.models.ResNet18_Weights.DEFAULT)
encoder.fc = nn.Identity()
encoder = encoder.float().eval().to(device)
MEAN = torch.tensor([0.485, 0.456, 0.406], dtype=torch.float32).view(1, 3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225], dtype=torch.float32).view(1, 3, 1, 1)

@torch.no_grad()
def extract_features(imgs, bs=128):
    out = []
    for i in range(0, len(imgs), bs):
        x = torch.from_numpy(imgs[i:i + bs]).float() / 255.0
        x = x.unsqueeze(1) if x.ndim == 3 else x.permute(0, 3, 1, 2)
        if x.shape[1] == 1:
            x = x.repeat(1, 3, 1, 1)
        x = nn.functional.interpolate(x, size=224, mode="bilinear", align_corners=False)
        out.append(encoder(((x - MEAN) / STD).to(device)).cpu().numpy())
    return np.concatenate(out).astype(np.float64)

DATA = {}
for flag in CFG["DATASETS"]:
    path = f"{CACHE}/{flag}_{CFG['IMG_SIZE']}.npz"
    if os.path.exists(path):
        d = np.load(path); F_pool, y_pool, F_shift, y_shift = d["Fp"], d["yp"], d["Fs"], d["ys"]
    else:
        t0 = time.time()
        Xtr, ytr = load_split(flag, "train"); Xva, yva = load_split(flag, "val"); Xte, yte = load_split(flag, "test")
        F_pool, y_pool = extract_features(np.concatenate([Xtr, Xva])), np.concatenate([ytr, yva])
        F_shift, y_shift = extract_features(Xte), yte
        np.savez(path, Fp=F_pool, yp=y_pool, Fs=F_shift, ys=y_shift)
        print(f"{flag}: features in {time.time() - t0:.0f}s on {device}")
    idx = np.arange(len(y_pool))
    r_, i_test = train_test_split(idx, test_size=CFG["TEST_FRAC"], stratify=y_pool, random_state=0)
    r_, i_cal = train_test_split(r_, test_size=CFG["CAL_FRAC"] / (1 - CFG["TEST_FRAC"]), stratify=y_pool[r_], random_state=0)
    i_tr, i_val = train_test_split(r_, test_size=CFG["VAL_FRAC"] / (1 - CFG["TEST_FRAC"] - CFG["CAL_FRAC"]),
                                   stratify=y_pool[r_], random_state=0)
    if QUICK_TEST:
        i_tr = i_tr[:600]
    F = {"train": F_pool[i_tr], "val": F_pool[i_val], "cal": F_pool[i_cal], "test": F_pool[i_test], "shift": F_shift}
    Y = {"train": y_pool[i_tr], "val": y_pool[i_val], "cal": y_pool[i_cal], "test": y_pool[i_test], "shift": y_shift}
    sc1 = StandardScaler().fit(F["train"])
    pca = PCA(n_components=NQ, random_state=0).fit(sc1.transform(F["train"]))
    sc2 = StandardScaler().fit(pca.transform(sc1.transform(F["train"])))
    Z = {k: sc2.transform(pca.transform(sc1.transform(v))) for k, v in F.items()}
    A = {k: np.clip(v, -3, 3) * (np.pi / 6) + np.pi / 2 for k, v in Z.items()}     # [0, pi]: no wrap-around
    DATA[flag] = dict(Z=Z, A=A, Y=Y, labels=INFO[flag]["label"])
    print(f"\n{flag}  labels={INFO[flag]['label']}  PCA variance kept={pca.explained_variance_ratio_.sum():.2f}")
    for k in Y:
        print(f"   {k:5s} n={len(Y[k]):5d}  positive share={Y[k].mean():.2f}")

## 3. Circuits and noise
Noise channels (depolarizing with probability `p`, amplitude damping with `g`) follow every gate on every qubit it touches. Readout error and shots are applied to the measured outcome exactly (VQCs) or to the kernel estimate (QSVM, shots only).

In [ ]:
def _noise(w, p, g):
    if p > 0:
        qml.DepolarizingChannel(p, wires=w)
    if g > 0:
        qml.AmplitudeDamping(g, wires=w)

def vqc_circuit(x, W, p=0.0, g=0.0, reupload=False):
    for l in range(NL):
        if l == 0 or reupload:
            for q in range(NQ):
                qml.RY(x[..., q], wires=q); _noise(q, p, g)
        for q in range(NQ):
            qml.Rot(W[l, q, 0], W[l, q, 1], W[l, q, 2], wires=q); _noise(q, p, g)
        for q in range(NQ):
            t = (q + 1) % NQ
            qml.CNOT(wires=[q, t]); _noise(q, p, g); _noise(t, p, g)
    return qml.expval(qml.PauliZ(0))

def feature_map(x, p=0.0, g=0.0):
    for _ in range(2):
        for q in range(NQ):
            qml.RY(x[..., q], wires=q); _noise(q, p, g)
            qml.RZ(x[..., q], wires=q); _noise(q, p, g)
        for q in range(NQ - 1):
            qml.CNOT(wires=[q, q + 1]); _noise(q, p, g); _noise(q + 1, p, g)

dev_ideal = qml.device("default.qubit", wires=NQ)
dev_mixed = qml.device("default.mixed", wires=NQ)
qn_vqc_train = qml.QNode(vqc_circuit, dev_ideal, interface="torch", diff_method="backprop")
qn_vqc_noisy = qml.QNode(vqc_circuit, dev_mixed, interface="numpy")

@qml.qnode(dev_ideal, interface="numpy")
def qn_state(x):
    feature_map(x)
    return qml.state()

@qml.qnode(dev_mixed, interface="numpy")
def qn_dm(x, p, g):
    feature_map(x, p, g)
    return qml.density_matrix(wires=range(NQ))

def batched(fn, X, *args):
    try:
        return np.asarray(fn(X, *args))
    except Exception:
        return np.stack([np.asarray(fn(x, *args)) for x in X])

def apply_readout(f, e0, e1):
    return f * (1 - e0 - e1) + (e1 - e0)

def apply_shots_expval(f, shots, rng):
    return 1 - 2 * rng.binomial(shots, np.clip((1 - f) / 2, 0, 1)) / shots

def cond(p=0.0, g=0.0, e0=0.0, e1=0.0, shots=None):
    return dict(p=p, g=g, e0=e0, e1=e1, shots=shots)
CLEAN = cond()

## 4. Models
Every model exposes the same interface: `clean(split)` gives noise-free scores (logits or SVM margins) and `observe(split, condition, rng)` gives scores under a noise condition. Calibrators only see these scores.

In [ ]:
def train_torch(model, Xtr, ytr, Xva, yva, seed):
    torch.manual_seed(seed)
    Xtr, ytr = torch.tensor(Xtr), torch.tensor(ytr, dtype=torch.float64)
    Xva, yva = torch.tensor(Xva), torch.tensor(yva, dtype=torch.float64)
    opt = torch.optim.Adam(model.parameters(), lr=CFG["LR"])
    lossf = nn.BCEWithLogitsLoss()
    best, state, bad = np.inf, None, 0
    for ep in range(CFG["EPOCHS"]):
        perm = torch.randperm(len(ytr))
        for i in range(0, len(ytr), CFG["BATCH"]):
            j = perm[i:i + CFG["BATCH"]]
            opt.zero_grad(); lossf(model(Xtr[j]), ytr[j]).backward(); opt.step()
        with torch.no_grad():
            vl = lossf(model(Xva), yva).item()
        if vl < best - 1e-4:
            best, bad, state = vl, 0, {k: v.detach().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= CFG["PATIENCE"]:
                break
    model.load_state_dict(state)
    return model, best, ep + 1

class VQCNet(nn.Module):
    def __init__(self, seed, reupload):
        super().__init__()
        g = torch.Generator().manual_seed(seed)
        self.W = nn.Parameter(torch.rand(NL, NQ, 3, generator=g) * 2 * np.pi)
        self.a = nn.Parameter(torch.tensor(2.0)); self.b = nn.Parameter(torch.tensor(0.0))
        self.reupload = reupload
    def forward(self, x):
        return self.a * qn_vqc_train(x, self.W, 0.0, 0.0, self.reupload) + self.b

class VQCModel:
    kind = "quantum"
    def __init__(self, name, D, seed, reupload):
        self.name, self.D, self.reupload = name, D, reupload
        net, self.val_loss, self.epochs = train_torch(VQCNet(seed, reupload), D["A"]["train"], D["Y"]["train"],
                                                      D["A"]["val"], D["Y"]["val"], seed)
        self.W = net.W.detach().numpy(); self.a, self.b = net.a.item(), net.b.item()
        with torch.no_grad():
            self._f = {(s, 0.0, 0.0): qn_vqc_train(torch.tensor(D["A"][s]), net.W, 0.0, 0.0, reupload).detach().numpy()
                       for s in ["val", "cal", "test", "shift"]}
    def exact(self, split, p, g):
        key = (split, p, g)
        if key not in self._f:
            self._f[key] = batched(lambda X, *a: qn_vqc_noisy(X, self.W, p, g, self.reupload), self.D["A"][split]).astype(float)
        return self._f[key]
    def clean(self, split):
        return self.a * self.exact(split, 0.0, 0.0) + self.b
    def observe(self, split, c, rng):
        f = self.exact(split, c["p"], c["g"])
        if c["e0"] or c["e1"]:
            f = apply_readout(f, c["e0"], c["e1"])
        if c["shots"]:
            f = apply_shots_expval(f, c["shots"], rng)
        return self.a * f + self.b

class QSVMModel:
    kind = "quantum"
    def __init__(self, name, D, seed):
        self.name, self.D = name, D
        Ytr = D["Y"]["train"]
        n = min(CFG["QSVM_N_TRAIN"], len(Ytr))
        self.idx = train_test_split(np.arange(len(Ytr)), train_size=n, stratify=Ytr, random_state=seed)[0] if n < len(Ytr) else np.arange(len(Ytr))
        self.Xtr = D["A"]["train"][self.idx]
        self._sv = {"train": batched(qn_state, self.Xtr)}
        for s in ["val", "cal", "test", "shift"]:
            self._sv[s] = batched(qn_state, D["A"][s])
        K = np.abs(self._sv["train"].conj() @ self._sv["train"].T) ** 2
        self.clf = SVC(C=CFG["QSVM_C"], kernel="precomputed").fit(K, Ytr[self.idx])
        self._dm = {}
    def _dms(self, split, p, g):
        key = (split, p, g)
        if key not in self._dm:
            X = self.Xtr if split == "train" else self.D["A"][split]
            self._dm[key] = batched(qn_dm, X, p, g).reshape(len(X), -1)
        return self._dm[key]
    def kernel(self, split, p, g):
        if p == 0 and g == 0:
            K = np.abs(self._sv[split].conj() @ self._sv["train"].T) ** 2
        else:
            K = np.real(self._dms(split, p, g) @ self._dms("train", p, g).conj().T)
        return np.clip(np.nan_to_num(K), 0.0, 1.0)      # rounding can give 1.0000000000000002
    def clean(self, split):
        return self.clf.decision_function(self.kernel(split, 0.0, 0.0))
    def observe(self, split, c, rng):
        K = self.kernel(split, c["p"], c["g"])          # readout error not modelled for the kernel proxy
        if c["shots"]:
            K = rng.binomial(c["shots"], K) / c["shots"]
        return self.clf.decision_function(K)

class ClassicalModel:
    kind = "classical"
    def __init__(self, name, D, seed):
        self.name, self.D = name, D
        if name == "MLP":
            torch.manual_seed(seed)
            net = nn.Sequential(nn.Linear(NQ, 6), nn.Tanh(), nn.Linear(6, 1), nn.Flatten(0))
            net, self.val_loss, self.epochs = train_torch(net, D["Z"]["train"], D["Y"]["train"], D["Z"]["val"], D["Y"]["val"], seed)
            with torch.no_grad():
                self._z = {s: net(torch.tensor(D["Z"][s])).numpy() for s in ["val", "cal", "test", "shift"]}
        else:
            svm = SVC(C=1.0, kernel="rbf", gamma="scale", random_state=seed).fit(D["Z"]["train"], D["Y"]["train"])
            self._z = {s: svm.decision_function(D["Z"][s]) for s in ["val", "cal", "test", "shift"]}
    def clean(self, split):
        return self._z[split]
    def observe(self, split, c, rng):
        return self._z[split]

class ControlModel:
    """Classical MLP whose logits receive the same stretch, shift and noise that the VQC's logits
    received under the same condition. If it behaves like the VQC, the effects come from the
    noise physics, not from the quantum classifier itself."""
    kind = "control"
    def __init__(self, name, mlp, vqc):
        self.name, self.mlp, self.vqc = name, mlp, vqc
    def clean(self, split):
        return self.mlp.clean(split)
    def observe(self, split, c, rng):
        zq = self.vqc.observe(split, c, rng); z0 = self.vqc.clean(split)
        s, off = np.polyfit(z0, zq, 1)
        sd = np.std(zq - (s * z0 + off))
        return s * self.mlp.clean(split) + off + rng.normal(0, sd, len(z0))

## 5. Metrics and calibration methods

In [ ]:
sigmoid = lambda z: 1 / (1 + np.exp(-np.clip(z, -50, 50)))
nll_logits = lambda z, y: np.mean(np.logaddexp(0, z) - y * z)

def ece_score(p, y, n_bins=CFG["N_BINS"]):
    b = np.minimum((p * n_bins).astype(int), n_bins - 1)
    return sum((b == k).mean() * abs(p[b == k].mean() - y[b == k].mean()) for k in range(n_bins) if (b == k).any())

def metrics(p, y):
    p = np.clip(p, 1e-6, 1 - 1e-6)
    return dict(Brier=np.mean((p - y) ** 2), NLL=-np.mean(y * np.log(p) + (1 - y) * np.log(1 - p)),
                ECE=ece_score(p, y), AUROC=roc_auc_score(y, p) if 0 < y.mean() < 1 else np.nan,
                Acc=np.mean((p >= 0.5) == y), CITL=p.mean() - y.mean(), n=len(y))

def fit_platt(z, y):
    a_, b_ = minimize(lambda v: nll_logits(v[0] * z + v[1], y), x0=[1.0, 0.0], method="BFGS").x
    return lambda zz: sigmoid(a_ * zz + b_)

def fit_temperature(z, y):
    T = np.exp(minimize_scalar(lambda lt: nll_logits(z / np.exp(lt), y), bounds=(-6, 6), method="bounded").x)
    return lambda zz: sigmoid(zz / T)

def em_prior(p, pi_s, iters=200):
    """Saerens et al. (2002): re-estimate the class prior on unlabeled data and adjust probabilities."""
    p = np.clip(p, 1e-6, 1 - 1e-6); pi_t = pi_s
    for _ in range(iters):
        num = pi_t / pi_s * p; q = num / (num + (1 - pi_t) / (1 - pi_s) * (1 - p))
        new = float(np.clip(q.mean(), 0.01, 0.99))
        if abs(new - pi_t) < 1e-7:
            break
        pi_t = new
    return q

def prior_adjust(p, pi_s, pi_t):
    p = np.clip(p, 1e-6, 1 - 1e-6); num = pi_t / pi_s * p
    return num / (num + (1 - pi_t) / (1 - pi_s) * (1 - p))

def eta(c):
    return np.array([c["p"], c["e0"], c["e1"]])

def fit_noise_conditioned_platt(zs, cs, y):
    raw = np.array([eta(c) for c in cs]); mu, sd = raw.mean(0), raw.std(0) + 1e-12
    Phi = np.vstack([np.repeat(((eta(c) - mu) / sd)[None], len(z), 0) for z, c in zip(zs, cs)])
    zz, yy = np.concatenate(zs), np.tile(y, len(zs))
    w = minimize(lambda w: nll_logits((w[0] + Phi @ w[1:4]) * zz + w[4] + Phi @ w[5:8], yy),
                 x0=np.r_[1.0, 0, 0, 0, 0.0, 0, 0, 0], method="BFGS").x
    def predict(z, c):
        ph = (eta(c) - mu) / sd
        return sigmoid((w[0] + ph @ w[1:4]) * z + w[4] + ph @ w[5:8])
    return predict

def fit_device_response(z0, zs, cs):
    """Physics-informed: score_obs ~= s(eta)*score_clean + c(eta), with
    log s = k*log(1-4p/3) + m*log(1-e0-e1)  and  c = c2*(1-s) + c1*(e1-e0).
    Fitted on UNLABELED calibration images (paired noise-free vs noisy scores) at the training conditions."""
    S, O = [], []
    for z in zs:
        s, o = np.polyfit(z0, z, 1); S.append(max(s, 1e-4)); O.append(o)
    S, O = np.array(S), np.array(O)
    X1 = np.array([[np.log(1 - 4 * c["p"] / 3), np.log(1 - c["e0"] - c["e1"])] for c in cs])
    k, m = np.linalg.lstsq(X1, np.log(S), rcond=None)[0]
    X2 = np.column_stack([1 - S, [c["e1"] - c["e0"] for c in cs]])
    c2, c1 = np.linalg.lstsq(X2, O, rcond=None)[0]
    def correct(z, c):
        s = np.exp(k * np.log(max(1 - 4 * c["p"] / 3, 1e-6)) + m * np.log(max(1 - c["e0"] - c["e1"], 1e-6)))
        s = max(s, 0.02)
        return (z - (c2 * (1 - s) + c1 * (c["e1"] - c["e0"]))) / s
    return correct, dict(k=k, m=m, c1=c1, c2=c2)

def moment_match(z, ref_mean, ref_std):
    return (z - z.mean()) / (z.std() + 1e-12) * ref_std + ref_mean

## 6. Noise conditions
Part B training conditions: depolarizing levels `FIT_P` × random readout. Test conditions: **held-out** depolarizing levels × new random readout, and an **unseen family**: amplitude damping on top of a small depolarizing level. Methods that use noise rates receive *reported* rates that are off by a random factor, and the amplitude damping is not reported at all.

In [ ]:
rc = np.random.default_rng(CFG["COND_SEED"])
EM_, SH = CFG["READOUT_MAX"], CFG["DRIFT_SHOTS"]
TRAIN_CONDS = [cond(p, 0.0, rc.uniform(0, EM_), rc.uniform(0, EM_), SH) for p in CFG["FIT_P"] for _ in range(CFG["N_READOUT_TRAIN"])]
TEST_CONDS = [("held-out depolarizing", cond(p, 0.0, rc.uniform(0, EM_), rc.uniform(0, EM_), SH))
              for p in CFG["HELDOUT_P"] for _ in range(CFG["N_READOUT_TEST"])]
TEST_CONDS += [("unseen: amplitude damping", cond(CFG["UNSEEN_BASE_P"], gm, rc.uniform(0, EM_), rc.uniform(0, EM_), SH))
               for gm in CFG["UNSEEN_AMP"] for _ in range(CFG["N_READOUT_TEST"])]
def reported(c):
    f = lambda v: v * np.exp(rc.normal(0, CFG["REPORT_ERROR"]))
    return cond(f(c["p"]), 0.0, f(c["e0"]), f(c["e1"]), c["shots"])     # amplitude damping is never reported
TEST_REPORTED = [reported(c) for _, c in TEST_CONDS]
REF = cond(**CFG["REF"], shots=SH)
COND_A = [("clean", "0", CLEAN)]
COND_A += [("depolarizing", str(p), cond(p=p)) for p in CFG["A_DEPOL"]]
COND_A += [("amplitude damping", str(gm), cond(g=gm)) for gm in CFG["A_AMP"]]
COND_A += [("shots", str(S), cond(shots=S)) for S in CFG["A_SHOTS"]]
COND_A += [("readout", f"{e0}/{e1}", cond(e0=e0, e1=e1)) for e0, e1 in CFG["A_READOUT"]]
COND_A += [("combined", "realistic", cond(0.005, 0.005, 0.01, 0.05, 1000))]
JOINT = cond(**CFG["JOINT"], shots=SH)
print(f"Part B: {len(TRAIN_CONDS)} training conditions, {len(TEST_CONDS)} test conditions")

## 7. Run all experiments
For each dataset and seed: train all five models, apply the AUROC gate, then run Parts A–D. Progress and timings are printed.

In [ ]:
def label_shift_idx(y, pi_t, rng):
    pos, neg = np.flatnonzero(y == 1), np.flatnonzero(y == 0)
    n = int(min(len(y), len(pos) / pi_t, len(neg) / (1 - pi_t)))
    k = int(round(n * pi_t))
    return np.r_[rng.choice(pos, k, replace=False), rng.choice(neg, n - k, replace=False)]

SAN, RA, RB, RC, RD, PARAMS = [], [], [], [], [], []
t_all = time.time()
for flag in CFG["DATASETS"]:
    D = DATA[flag]; Y = D["Y"]; pi_s = Y["cal"].mean()
    for seed in CFG["SEEDS"]:
        print(f"\n===== {flag} | seed {seed} =====")
        t0 = time.time()
        models = {"VQC": VQCModel("VQC", D, seed, False),
                  "Re-uploading VQC": VQCModel("Re-uploading VQC", D, seed, True),
                  "QSVM": QSVMModel("QSVM", D, seed),
                  "MLP": ClassicalModel("MLP", D, seed),
                  "RBF-SVM": ClassicalModel("RBF-SVM", D, seed)}
        print(f"  trained 5 models in {time.time() - t0:.0f}s")
        # ---- AUROC gate on the validation set ----
        ok = {}
        for name, m in models.items():
            auc_val = roc_auc_score(Y["val"], m.clean("val"))
            cal = fit_platt(m.clean("cal"), Y["cal"])
            met = metrics(cal(m.clean("test")), Y["test"])
            ok[name] = auc_val >= CFG["AUROC_GATE"]
            SAN.append(dict(dataset=flag, seed=seed, model=name, val_AUROC=auc_val, test_AUROC=met["AUROC"],
                            test_Brier_Platt=met["Brier"], test_ECE_Platt=met["ECE"], passed_gate=ok[name]))
            print(f"  {name:17s} val AUROC={auc_val:.3f}  {'OK' if ok[name] else '*** EXCLUDED: below AUROC gate ***'}")
        if ok["VQC"] and ok["MLP"]:
            models["MLP + VQC distortion"] = ControlModel("MLP + VQC distortion", models["MLP"], models["VQC"]); ok["MLP + VQC distortion"] = True
        rng = np.random.default_rng(1000 + seed)
        hw = [n for n, m in models.items() if m.kind in ("quantum", "control") and ok[n]]

        # ---- Part A: one noise type at a time (quantum models) ----
        t0 = time.time()
        for name in [n for n in hw if models[n].kind == "quantum"]:
            m = models[name]
            for fam, lvl, c in COND_A:
                zc, zt = m.observe("cal", c, rng), m.observe("test", c, rng)
                meths = {"Temperature (refit)": fit_temperature(zc, Y["cal"])(zt), "Platt (refit)": fit_platt(zc, Y["cal"])(zt)}
                if name != "QSVM":
                    meths["Uncalibrated"] = sigmoid(zt)
                for meth, pr in meths.items():
                    RA.append(dict(dataset=flag, seed=seed, model=name, family=fam, level=lvl, method=meth, **metrics(pr, Y["test"])))
        print(f"  Part A done ({time.time() - t0:.0f}s)")

        # ---- Part B: transfer to unseen noise ----
        t0 = time.time()
        for name in hw:
            m = models[name]
            fixed = fit_platt(m.observe("cal", REF, rng), Y["cal"])
            z_ref = m.observe("cal", REF, rng); ref_mean, ref_std = z_ref.mean(), z_ref.std()
            z_train = [m.observe("cal", c, rng) for c in TRAIN_CONDS]
            ncp = fit_noise_conditioned_platt(z_train, TRAIN_CONDS, Y["cal"])
            correct, prm = fit_device_response(m.clean("cal"), z_train, TRAIN_CONDS)
            PARAMS.append(dict(dataset=flag, seed=seed, model=name, **prm))
            clean_platt = fit_platt(m.clean("cal"), Y["cal"])
            for ci, ((fam, c), crep) in enumerate(zip(TEST_CONDS, TEST_REPORTED)):
                oracle = fit_platt(m.observe("cal", c, rng), Y["cal"])
                for ev in ["test", "shift"]:
                    z = m.observe(ev, c, rng)
                    preds = {"Fixed Platt": fixed(z), "Oracle Platt": oracle(z),
                             "Noise-conditioned Platt": ncp(z, crep),
                             "Physics-informed": clean_platt(correct(z, crep)),
                             "Moment matching": fixed(moment_match(z, ref_mean, ref_std))}
                    if name not in ("QSVM",):
                        preds["Uncalibrated"] = sigmoid(z)
                    for meth, pr in preds.items():
                        RB.append(dict(dataset=flag, seed=seed, model=name, family=fam, cond=ci, p=c["p"], g=c["g"],
                                       e0=c["e0"], e1=c["e1"], eval_set=ev, method=meth, **metrics(pr, Y[ev])))
                # ---- Part D: moment matching with small unlabeled batches (VQC, held-out depolarizing) ----
                if name == "VQC" and fam == "held-out depolarizing":
                    z = m.observe("test", c, rng); order = np.random.default_rng(ci).permutation(len(z))
                    for bs in CFG["BATCH_SIZES"] + ["all"]:
                        pr = np.empty(len(z))
                        n_batches = 1 if bs == "all" else max(1, len(z) // bs)
                        for j in np.array_split(order, n_batches):      # near-equal batches of at least `bs` images
                            pr[j] = fixed(moment_match(z[j], ref_mean, ref_std))
                        RD.append(dict(dataset=flag, seed=seed, cond=ci, p=c["p"], batch=str(bs), **metrics(pr, Y["test"])))
        print(f"  Part B + D done ({time.time() - t0:.0f}s)")

        # ---- Part C: label shift, with and without noise ----
        t0 = time.time()
        lrng = np.random.default_rng(CFG["COND_SEED"])
        subsets = {"source": np.arange(len(Y["test"]))}
        for pt in CFG["PRIORS"]:
            subsets[f"{pt:.2f}"] = label_shift_idx(Y["test"], pt, lrng)
        for name, m in models.items():
            if not ok[name] or m.kind == "control":
                continue
            settings = [("no noise", CLEAN)] + ([("noise", JOINT)] if m.kind == "quantum" else [])
            for sname, c in settings:
                fixed = fit_platt(m.observe("cal", REF if m.kind == "quantum" else CLEAN, rng), Y["cal"])
                z_ref = m.observe("cal", REF if m.kind == "quantum" else CLEAN, rng)
                if m.kind == "quantum":
                    z_train = [m.observe("cal", cc, rng) for cc in TRAIN_CONDS]
                    correct, _ = fit_device_response(m.clean("cal"), z_train, TRAIN_CONDS)
                    crep = reported(c)
                else:
                    correct, crep = (lambda z, _c: z), c
                clean_platt = fit_platt(m.clean("cal"), Y["cal"])
                oracle = fit_platt(m.observe("cal", c, rng), Y["cal"])
                z_full = m.observe("test", c, rng)
                for sub, idx in subsets.items():
                    z, y = z_full[idx], Y["test"][idx]
                    pi_t = y.mean()
                    pi_corr = clean_platt(correct(z, crep))
                    mm = fixed(moment_match(z, z_ref.mean(), z_ref.std()))
                    preds = {"Fixed Platt": fixed(z), "Physics-informed": pi_corr, "Physics-informed + EM prior": em_prior(pi_corr, pi_s),
                             "Moment matching": mm, "Moment matching + EM prior": em_prior(mm, pi_s),
                             "Oracle (noise + true prior)": prior_adjust(oracle(z), pi_s, pi_t)}
                    for meth, pr in preds.items():
                        RC.append(dict(dataset=flag, seed=seed, model=name, setting=sname, target_prior=sub,
                                       true_prior=pi_t, method=meth, **metrics(pr, y)))
        print(f"  Part C done ({time.time() - t0:.0f}s)")

san, resA, resB, resC, resD = map(pd.DataFrame, [SAN, RA, RB, RC, RD])
OUT = "/content/v4_results" if os.path.isdir("/content") else "v4_results"
os.makedirs(OUT, exist_ok=True)
for n_, df in [("sanity", san), ("partA", resA), ("partB", resB), ("partC", resC), ("partD", resD), ("device_fit", pd.DataFrame(PARAMS))]:
    df.to_csv(f"{OUT}/v4_{n_}.csv", index=False)
json.dump({k: v for k, v in CFG.items()}, open(f"{OUT}/v4_config.json", "w"), indent=1, default=str)
print(f"\nAll done in {(time.time() - t_all) / 60:.1f} min. Results in {OUT}/")

## 8. Sanity check and gate
All models should be close to each other on noise-free data. Any row with `passed_gate = False` was excluded from Parts A–D.

In [ ]:
display(san.groupby(["dataset", "model"])[["val_AUROC", "test_AUROC", "test_Brier_Platt", "test_ECE_Platt", "passed_gate"]].mean().round(3))

## 9. Part A: damage and repair, relative to the noise-free model
Δ = metric under noise − metric with no noise (same model, method and seed). Brier is the primary metric. Near 0 = no damage left.

In [ ]:
if len(resA):
    base = resA[resA.family == "clean"].set_index(["dataset", "seed", "model", "method"])
    for met in ["Brier", "ECE"]:
        resA["d" + met] = resA[met].values - base.loc[list(zip(resA.dataset, resA.seed, resA.model, resA.method)), met].values
    for flag in resA.dataset.unique():
        print(f"\n===== {flag}: ΔBrier (mean over seeds) =====")
        d = resA[(resA.dataset == flag) & (resA.family != "clean")]
        display(d.groupby(["model", "family", "level", "method"], sort=False)["dBrier"].mean().unstack("method").round(4))
        print(f"AUROC under each condition (ranking ability):")
        display(d[d.method == "Platt (refit)"].groupby(["model", "family", "level"], sort=False)["AUROC"].mean().unstack("model").round(3))

## 10. Part B: calibration transfer to unseen noise (main result)
`gap_closed` = share of the Fixed → Oracle improvement in Brier score that a method recovers (1 = as good as refitting with new labels, 0 = no better than never updating).

In [ ]:
ORDER = ["Uncalibrated", "Fixed Platt", "Noise-conditioned Platt", "Physics-informed", "Moment matching", "Oracle Platt"]
def table_B(df, metric="Brier"):
    g = df.groupby(["dataset", "model", "family", "seed", "method"])[metric].mean().reset_index()
    t = g.groupby(["dataset", "model", "family", "method"])[metric].mean().unstack("method")
    t = t[[c for c in ORDER if c in t.columns]]
    gap = (t["Fixed Platt"].values[:, None] - t.values) / (t["Fixed Platt"] - t["Oracle Platt"]).values[:, None]
    return t.round(4), pd.DataFrame(gap, index=t.index, columns=t.columns).round(2)

if len(resB):
    for ev in ["test", "shift"]:
        tB, gB = table_B(resB[resB.eval_set == ev])
        print(f"\n===== Brier score, {'matched test' if ev == 'test' else 'shift'} set =====")
        display(tB)
        print("Share of the Fixed -> Oracle gap closed:")
        display(gB.drop(columns=[c for c in ["Uncalibrated", "Fixed Platt", "Oracle Platt"] if c in gB.columns]))

### 10b. Paired comparison with Fixed Platt (matched test set, Brier)
Per condition and seed: Brier(method) − Brier(Fixed Platt). Negative = better than never updating. 95% CI from a bootstrap over condition × seed pairs.

In [ ]:
if len(resB):
    rows, rb = [], np.random.default_rng(0)
    d = resB[resB.eval_set == "test"]
    for (flag, name, fam), g in d.groupby(["dataset", "model", "family"]):
        pv = g.pivot_table(index=["seed", "cond"], columns="method", values="Brier")
        for meth in ["Noise-conditioned Platt", "Physics-informed", "Moment matching", "Oracle Platt"]:
            x = (pv[meth] - pv["Fixed Platt"]).values
            bs = [x[rb.integers(0, len(x), len(x))].mean() for _ in range(CFG["N_BOOT"])]
            lo, hi = np.percentile(bs, [2.5, 97.5])
            rows.append(dict(dataset=flag, model=name, family=fam, method=meth, mean_diff=round(x.mean(), 4),
                             CI=f"[{lo:.4f}, {hi:.4f}]", verdict="better" if hi < 0 else ("worse" if lo > 0 else "no clear difference")))
    compB = pd.DataFrame(rows); compB.to_csv(f"{OUT}/v4_partB_vs_fixed.csv", index=False)
    display(compB)

### 10c. Figure: Brier score vs held-out depolarizing level (matched test set)

In [ ]:
if len(resB):
    for flag in resB.dataset.unique():
        d = resB[(resB.dataset == flag) & (resB.eval_set == "test") & (resB.family == "held-out depolarizing")]
        names = list(d.model.unique())
        fig, axes = plt.subplots(1, len(names), figsize=(4.3 * len(names), 3.8), sharey=True, squeeze=False)
        for ax, name in zip(axes[0], names):
            for meth in [m_ for m_ in ORDER if m_ != "Uncalibrated"]:
                s_ = d[(d.model == name) & (d.method == meth)].groupby("p")["Brier"].mean()
                ax.plot(s_.index, s_.values, "k-" if meth == "Oracle Platt" else "o-", label=meth)
            ax.set(xscale="log", title=name, xlabel="depolarizing p")
        axes[0][0].set_ylabel("Brier (lower is better)"); axes[0][0].legend(fontsize=7)
        fig.suptitle(flag); plt.tight_layout(); plt.savefig(f"{OUT}/v4_partB_{flag}.png", dpi=150); plt.show()

## 11. Part C: label shift, alone and with noise
`target_prior` = positive share of the test subset (`source` = unchanged). Methods without "EM prior" assume the source prior. The oracle knows both the noise (refit with labels) and the true prior: a best case, not a usable method.

In [ ]:
if len(resC):
    for flag in resC.dataset.unique():
        print(f"\n===== {flag}: Brier score under label shift =====")
        d = resC[resC.dataset == flag]
        display(d.groupby(["model", "setting", "target_prior", "method"], sort=False)["Brier"].mean().unstack("method").round(4))
        print("Subset sizes:", d.groupby("target_prior")["n"].first().to_dict())

## 12. Part D: moment matching with small unlabeled batches (VQC, matched test set)

In [ ]:
if len(resD):
    display(resD.groupby(["dataset", "batch"])[["Brier", "ECE", "NLL"]].mean().round(4))

## 13. How to read the results
* **Sanity (8):** if a model is excluded or far below the others, don't interpret its rows.
* **Part A (9):** expect refitted Platt to repair readout and depolarizing noise almost exactly. Amplitude damping is *non-unital* (it pushes qubits toward |0⟩): if refitted Platt leaves damage there, that is a genuine new finding.
* **Part B (10):** the main question is whether the label-free methods stay close to the oracle on the **unseen amplitude-damping family**, where the noise-rate-based methods are given incomplete information. Compare the VQC row with **MLP + VQC distortion**: if they match, the effect comes from the noise physics, not the quantum classifier, and the paper should say so.
* **Part C (11):** the identifiability test. Moment matching treats every change in the output distribution as noise; under label shift it will mis-correct. The physics-informed correction uses reported noise rates, so its noise correction should not be confused by label shift, and EM can then fix the prior. If "Physics-informed + EM prior" approaches the oracle while "Moment matching + EM prior" does not, that supports the shift-disentanglement idea.
* **BreastMNIST** has ~95 calibration and ~95 test images; treat its results as exploratory.

**Limitations:** simulated noise only (no device noise model or hardware); the QSVM noise uses a noisy-state overlap proxy and no readout error; the physics-informed method is fitted with paired noise-free and noisy scores on unlabeled images, which on hardware would require a device-characterization step; seeds vary initialisation and measurement randomness, not the data split.

In [ ]:
import shutil
archive = shutil.make_archive(OUT, "zip", OUT)
print("Results archive:", archive)
try:
    from google.colab import files
    files.download(archive)
except ImportError:
    pass